In [1]:
from importlib.metadata import version

print("pybaseball:", version("pybaseball"))
print("pandas:", version("pandas"))

pybaseball: 2.2.7
pandas: 3.0.5


In [2]:
import inspect
from pybaseball import statcast

print(inspect.signature(statcast))
print()
print(statcast.__doc__)

(start_dt: str = None, end_dt: str = None, team: str = None, verbose: bool = True, parallel: bool = True) -> pandas.DataFrame


    Pulls statcast play-level data from Baseball Savant for a given date range.

    INPUTS:
    start_dt: YYYY-MM-DD : the first date for which you want statcast data
    end_dt: YYYY-MM-DD : the last date for which you want statcast data
    team: optional (defaults to None) : city abbreviation of the team you want data for (e.g. SEA or BOS)
    verbose: bool (defaults to True) : whether to print updates on query progress
    parallel: bool (defaults to True) : whether to parallelize HTTP requests in large queries

    If no arguments are provided, this will return yesterday's statcast data.
    If one date is provided, it will return that date's statcast data.
    


In [3]:
import pybaseball

print([n for n in dir(pybaseball) if "cache" in n.lower()])

['__cached__', 'cache']


In [4]:
from pybaseball import cache
cache.enable()
print("cache enabled")

cache enabled


In [5]:
df = statcast(start_dt="2024-04-15", end_dt="2024-04-15")
print(df.shape)

This is a large query, it may take a moment to complete


100%|██████████| 1/1 [00:01<00:00,  1.24s/it]

(4362, 119)


In [6]:
cols = df.columns.tolist()
print(len(cols))
for c in cols:
    print(c)

119
pitch_type
game_date
release_speed
release_pos_x
release_pos_z
player_name
batter
pitcher
events
description
spin_dir
spin_rate_deprecated
break_angle_deprecated
break_length_deprecated
zone
des
game_type
stand
p_throws
home_team
away_team
type
hit_location
bb_type
balls
strikes
game_year
pfx_x
pfx_z
plate_x
plate_z
on_3b
on_2b
on_1b
outs_when_up
inning
inning_topbot
hc_x
hc_y
tfs_deprecated
tfs_zulu_deprecated
umpire
sv_id
vx0
vy0
vz0
ax
ay
az
sz_top
sz_bot
hit_distance_sc
launch_speed
launch_angle
effective_speed
release_spin_rate
release_extension
game_pk
fielder_2
fielder_3
fielder_4
fielder_5
fielder_6
fielder_7
fielder_8
fielder_9
release_pos_y
estimated_ba_using_speedangle
estimated_woba_using_speedangle
woba_value
woba_denom
babip_value
iso_value
launch_speed_angle
at_bat_number
pitch_number
pitch_name
home_score
away_score
bat_score
fld_score
post_away_score
post_home_score
post_bat_score
post_fld_score
if_fielding_alignment
of_fielding_alignment
spin_axis
delta_home_win_e

In [7]:
print(df.dtypes)
df.head()

pitch_type                                      str
game_date                                       str
release_speed                               Float64
release_pos_x                               Float64
release_pos_z                               Float64
                                             ...   
attack_angle                                Float64
attack_direction                            Float64
swing_path_tilt                             Float64
intercept_ball_minus_batter_pos_x_inches    Float64
intercept_ball_minus_batter_pos_y_inches    Float64
Length: 119, dtype: object


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
1887,ST,2024-04-15,85.8,-2.38,5.85,"Thompson, Keegan",572233,624522,strikeout,swinging_strike,...,1,2.94,-1.19,-1.19,41.2,-3.234095,15.366552,25.348516,60.493481,19.717626
1903,FF,2024-04-15,95.0,-2.63,5.79,"Thompson, Keegan",572233,624522,NaN,foul,...,1,1.4,-0.02,-0.02,40.3,4.764329,11.026102,23.22736,44.535909,25.914323
1977,ST,2024-04-15,85.1,-2.57,5.78,"Thompson, Keegan",572233,624522,NaN,ball,...,1,2.97,-1.1,-1.1,39.1,<NA>,<NA>,<NA>,<NA>,<NA>
1993,ST,2024-04-15,86.2,-2.52,5.84,"Thompson, Keegan",572233,624522,NaN,blocked_ball,...,1,3.11,-0.78,-0.78,39.9,<NA>,<NA>,<NA>,<NA>,<NA>
2055,FF,2024-04-15,94.6,-2.63,5.91,"Thompson, Keegan",572233,624522,NaN,called_strike,...,1,1.19,0.11,0.11,40.3,<NA>,<NA>,<NA>,<NA>,<NA>


In [8]:
miss = df.isna().mean().sort_values(ascending=False)
print(miss[miss > 0].head(30))

tfs_deprecated                     1.000000
spin_dir                           1.000000
break_length_deprecated            1.000000
break_angle_deprecated             1.000000
sv_id                              1.000000
umpire                             1.000000
tfs_zulu_deprecated                1.000000
spin_rate_deprecated               1.000000
on_3b                              0.908070
miss_distance                      0.906236
estimated_ba_using_speedangle      0.825539
estimated_slg_using_speedangle     0.825539
launch_speed_angle                 0.823017
hc_x                               0.822788
hc_y                               0.822788
bb_type                            0.822788
on_2b                              0.813388
hit_location                       0.773498
estimated_woba_using_speedangle    0.748968
des                                0.746217
woba_denom                         0.745759
events                             0.745530
woba_value                      

In [9]:
print(df.shape)

(4362, 119)


In [10]:
print(df["type"].value_counts())
print()
print(df.groupby("type")["launch_speed"].apply(lambda s: s.isna().mean()))

type
S    2038
B    1551
X     773
Name: count, dtype: int64

type
B    1.000000
S    0.657507
X    0.001294
Name: launch_speed, dtype: float64


In [11]:
print(df["description"].value_counts())

description
ball                       1436
foul                        798
hit_into_play               773
called_strike               758
swinging_strike             416
blocked_ball                 92
foul_tip                     37
swinging_strike_blocked      20
automatic_ball               16
foul_bunt                     8
hit_by_pitch                  7
missed_bunt                   1
Name: count, dtype: int64


In [12]:
from pathlib import Path
from datetime import date

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "CLAUDE.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

out_path = PROJECT_ROOT / "data" / "raw" / f"statcast_2024-04-15_ingested_{date.today().isoformat()}.parquet"

df.to_parquet(out_path, index=False)
print(out_path.name, round(out_path.stat().st_size / 1e6, 2), "MB")

statcast_2024-04-15_ingested_2026-08-31.parquet 0.82 MB
